In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms, models

# Configuration
DATA_DIR = "kaggle/data"  # Directory containing 'Class_0' and 'Class_1'
BATCH_SIZE = 32
IMAGE_SIZE = 224
LEARNING_RATE = 1e-4
EPOCHS = 10
TRAIN_RATIO = 0.8  # 80% train, 20% validation
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using {DEVICE}")

using cuda


In [2]:
# Train pipeline with augmentations
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Validation pipeline (no augmentations)
val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class TransformedSubset(Dataset):
    """Wrapper to apply distinct transformations to subsets after random_split"""
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

In [7]:
# Load full dataset without initial transforms
full_dataset = datasets.ImageFolder(root=DATA_DIR)

# Exclude 'sample' if ImageFolder picked it up as a class
valid_classes = ['Class_0', 'Class_1']
indices = [i for i, (_, label) in enumerate(full_dataset.samples) 
           if full_dataset.classes[label] in valid_classes]

filtered_dataset = torch.utils.data.Subset(full_dataset, indices)

# Compute train and val split lengths
train_size = int(TRAIN_RATIO * len(filtered_dataset))
val_size = len(filtered_dataset) - train_size

# Perform random split
train_subset, val_subset = random_split(filtered_dataset, [train_size, val_size])

# Apply specific transforms
train_data = TransformedSubset(train_subset, transform=train_transform)
val_data = TransformedSubset(val_subset, transform=val_transform)

# DataLoaders
# Update Section 3 DataLoaders:
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Dataset split complete: {train_size} training samples, {val_size} validation samples.")

Dataset split complete: 4430 training samples, 1108 validation samples.


In [8]:
def build_mobilenetv3_binary():
    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    
    # Freeze feature extraction backbone
    for param in model.features.parameters():
        param.requires_grad = False

    # Adjust final linear classifier for 1 binary logit output
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, 1)
    
    return model

model = build_mobilenetv3_binary().to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.classifier.parameters(), lr=LEARNING_RATE)

In [9]:
def train_and_validate(model, train_loader, val_loader, epochs):
    for epoch in range(epochs):
        # Training Phase
        model.train()
        running_loss, correct_train, total_train = 0.0, 0, 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.float().unsqueeze(1).to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct_train += (preds == labels).sum().item()
            total_train += labels.size(0)
            
        epoch_loss = running_loss / total_train
        epoch_acc = correct_train / total_train

        # Validation Phase
        model.eval()
        val_loss, correct_val, total_val = 0.0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.float().unsqueeze(1).to(DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                preds = (torch.sigmoid(outputs) >= 0.5).float()
                correct_val += (preds == labels).sum().item()
                total_val += labels.size(0)

        val_epoch_loss = val_loss / total_val
        val_epoch_acc = correct_val / total_val

        print(f"Epoch [{epoch+1}/{epochs}] | "
              f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} | "
              f"Val Loss: {val_epoch_loss:.4f} Acc: {val_epoch_acc:.4f}")

# Execute training run
train_and_validate(model, train_loader, val_loader, EPOCHS)

# Save checkpoint
torch.save(model.state_dict(), "mobilenetv3_oil_spill.pth")

Epoch [1/10] | Train Loss: 0.5180 Acc: 0.7619 | Val Loss: 0.4463 Acc: 0.8457
Epoch [2/10] | Train Loss: 0.4281 Acc: 0.8093 | Val Loss: 0.3761 Acc: 0.8682
Epoch [3/10] | Train Loss: 0.3890 Acc: 0.8379 | Val Loss: 0.3618 Acc: 0.8709
Epoch [4/10] | Train Loss: 0.3777 Acc: 0.8343 | Val Loss: 0.3480 Acc: 0.8727
Epoch [5/10] | Train Loss: 0.3477 Acc: 0.8542 | Val Loss: 0.3333 Acc: 0.8727
Epoch [6/10] | Train Loss: 0.3441 Acc: 0.8567 | Val Loss: 0.3140 Acc: 0.8782
Epoch [7/10] | Train Loss: 0.3373 Acc: 0.8571 | Val Loss: 0.2969 Acc: 0.8827
Epoch [8/10] | Train Loss: 0.3394 Acc: 0.8481 | Val Loss: 0.2767 Acc: 0.8935
Epoch [9/10] | Train Loss: 0.3271 Acc: 0.8625 | Val Loss: 0.2796 Acc: 0.8908
Epoch [10/10] | Train Loss: 0.3233 Acc: 0.8655 | Val Loss: 0.2841 Acc: 0.8836
